In [2]:
!pip install scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [3]:
import configparser, urllib.request, os, time
import boto3, sagemaker, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report


from sagemaker.inputs import TrainingInput
from sagemaker.amazon.amazon_estimator import image_uris
from sagemaker.serializers import CSVSerializer
from sagemaker.session import Session

#boto is a special package that allows python to talk to AWS services, including S3, EC2 and others. 

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


# Let's Setup BOTO

In [4]:
s3_client = boto3.client("s3")


BUCKET = "leviathan-s3"
region = 'us-west-2'

In [5]:
try:
    
    s3_client.create_bucket(
        Bucket=BUCKET, CreateBucketConfiguration={"LocationConstraint": region}
    )
    print("Created bucket", BUCKET)
except s3_client.exceptions.BucketAlreadyOwnedByYou:
    print("Using existing bucket", BUCKET)

Using existing bucket leviathan-s3


## Boto3 common commands

In [6]:
response = s3_client.list_buckets()
print("Available S3 Buckets:")
for bucket in response['Buckets']:
    print(f"  - {bucket['Name']} (Created: {bucket['CreationDate']})")

Available S3 Buckets:
  - leviathan-s3 (Created: 2026-10-19 16:22:14+00:00)


In [7]:
!touch empty.txt

In [8]:
local_file_path = 'empty.txt'
s3_key = 's3_empty_v2.txt' #  This is the file name  which  will be present inside the bucket.

s3_client.upload_file(local_file_path, BUCKET, s3_key)

In [9]:
DATA_URL = (
    "https://d1.awsstatic.com/tmt/build-train-deploy-machine-learning-model-"
    "sagemaker/bank_clean.27f01fbbdf43271788427f3682996ae29ceca05d.csv"
)


# Define local file path
LOCAL_CSV = "bank_clean.csv"


urllib.request.urlretrieve(DATA_URL, LOCAL_CSV)
print("Downloaded", LOCAL_CSV)


Downloaded bank_clean.csv


In [10]:
df = pd.read_csv(LOCAL_CSV, index_col=0)
print("Shape:", df.shape)
display(df.head(3))

Shape: (41188, 61)


,age,campaign,pdays,previous,no_previous_contact,not_working,job_admin.,job_blue-collar,job_entrepreneur,job_housemaid,...,day_of_week_fri,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_failure,poutcome_nonexistent,poutcome_success,y_no,y_yes
0,56,1,999,0,1,0,0,0,0,1,...,0,1,0,0,0,0,1,0,1,0
1,57,1,999,0,1,0,0,0,0,0,...,0,1,0,0,0,0,1,0,1,0
2,37,1,999,0,1,0,0,0,0,0,...,0,1,0,0,0,0,1,0,1,0


## Uploading training/testing data to S3, as that's where model will get it from in production

In [11]:
train_df, test_df = train_test_split(
    df, train_size=0.7, shuffle=True, random_state=1729
)

In [12]:
train_df.drop(["y_no", "y_yes"], axis=1).head()

,age,campaign,pdays,previous,no_previous_contact,not_working,job_admin.,job_blue-collar,job_entrepreneur,job_housemaid,...,month_oct,month_sep,day_of_week_fri,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_failure,poutcome_nonexistent,poutcome_success
28868,31,1,999,1,1,1,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
10545,29,1,999,0,1,0,1,0,0,0,...,0,0,0,0,0,1,0,0,1,0
25907,53,2,999,0,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
1289,60,1,999,0,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,1,0
23064,55,1,999,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0


In [13]:
pd.concat(
    [train_df["y_yes"], train_df.drop(["y_no", "y_yes"], axis=1)], axis=1
).head()

,y_yes,age,campaign,pdays,previous,no_previous_contact,not_working,job_admin.,job_blue-collar,job_entrepreneur,...,month_oct,month_sep,day_of_week_fri,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_failure,poutcome_nonexistent,poutcome_success
28868,0,31,1,999,1,1,1,0,0,0,...,0,0,0,0,1,0,0,1,0,0
10545,0,29,1,999,0,1,0,1,0,0,...,0,0,0,0,0,1,0,0,1,0
25907,0,53,2,999,0,1,0,0,0,1,...,0,0,0,0,0,0,1,0,1,0
1289,0,60,1,999,0,1,0,0,0,1,...,0,0,0,0,1,0,0,0,1,0
23064,0,55,1,999,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0


In [14]:
PREFIX = "xgboost-bank"

train_payload = pd.concat(
    [train_df["y_yes"], train_df.drop(["y_no", "y_yes"], axis=1)], axis=1
)


train_csv_path = "train.csv"

train_payload.to_csv(train_csv_path, index=False, header=False)



In [15]:
s3_train_key = f"{PREFIX}/train/{train_csv_path}"

print("S3 train key: ", s3_train_key)

s3_client.upload_file(train_csv_path, BUCKET, s3_train_key) 
# upload train_csv_path (local csv file, to bucket (leviathan-s3) at location s3_train_key


print(f"Uploaded training data to s3://{BUCKET}/{s3_train_key}")

S3 train key:  xgboost-bank/train/train.csv
Uploaded training data to s3://leviathan-s3/xgboost-bank/train/train.csv


In [16]:
test_payload = pd.concat(
    [test_df["y_yes"], test_df.drop(["y_no", "y_yes"], axis=1)], axis=1
)
test_csv_path = "test.csv"
test_payload.to_csv(test_csv_path, index=False, header=False)
s3_test_key = f"{PREFIX}/test/{test_csv_path}"
s3_client.upload_file(test_csv_path, BUCKET, s3_test_key)
print(f"Uploaded test data to s3://{BUCKET}/{s3_test_key}")

Uploaded test data to s3://leviathan-s3/xgboost-bank/test/test.csv


from xgboost import XGBClassifier

xgb = XGBClassifier()

xgb.fit(X_train, y_train)

xgb.score(X_test, y_test)

# Configuring and Setting up Xgboost Container

In [17]:

s3_train = TrainingInput(f"s3://{BUCKET}/{PREFIX}/train/", content_type="text/csv")
s3_val = TrainingInput(f"s3://{BUCKET}/{PREFIX}/test/", content_type="text/csv")

In [19]:
# Training Instance

# Training Model

container = image_uris.retrieve('xgboost',boto3.Session().region_name,'1.5-1')

# initialize hyperparameters
hyperparameters = {
        "max_depth":"5",
        "eta":"0.2",
        "gamma":"4",
        "min_child_weight":"6",
        "subsample":"0.7",
        "objective":"binary:logistic",
        "num_round":50
}

estimator = sagemaker.estimator.Estimator(image_uri=container, 
                                          hyperparameters=hyperparameters,
                                          role=sagemaker.get_execution_role(),
                                          instance_count=1, 
                                          instance_type='ml.m5.2xlarge', 
                                          volume_size=5, # 5 GB 
                                          output_path=f"s3://{BUCKET}/{PREFIX}/output")


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


In [20]:
print("Starting training …")

estimator.fit({"train": s3_train, "validation": s3_val}, wait=True)


INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-10-19-16-53-19-154


Starting training …
2026-10-19 16:53:20 Starting - Starting the training job...
2026-10-19 16:53:34 Starting - Preparing the instances for training...
2026-10-19 16:54:11 Downloading - Downloading the training image...
2026-10-19 16:54:51 Training - Training image download completed. Training in progress....
2026-10-19 16:55:16 Uploading - Uploading generated training model/miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-10-19 16:55:12.481 ip-10-0-215-253.us-west-2.compute.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-10-19 16:55:12.503 ip-10-0-215-253.us-west-2.compute.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-10-19:16:55:12:INFO] Imported framework sagemaker_xgboost_container.training
[2026-10-19:16:55:12:INF

In [21]:
# 6. Deploy Realtime Endpoint

print("Deploying endpoint … this may take a few minutes")
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.2xlarge",
    serializer=CSVSerializer(),
)
endpoint_name = predictor.endpoint_name
print("Endpoint active →", endpoint_name)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-10-19-16-57-09-047


Deploying endpoint … this may take a few minutes


INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-2026-10-19-16-57-09-047
INFO:sagemaker:Creating endpoint with name sagemaker-xgboost-2026-10-19-16-57-09-047


------!

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/base_predictor.py:140: SageMakerV2DeprecationWarning: Predictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Endpoint active → sagemaker-xgboost-2026-10-19-16-57-09-047


In [22]:
X_test = test_df.drop(["y_no", "y_yes"], axis=1).values
X_test

array([[ 54,   3, 999, ...,   0,   1,   0],
       [ 56,   2, 999, ...,   0,   1,   0],
       [ 32,   2, 999, ...,   0,   1,   0],
       ...,
       [ 58,   2, 999, ...,   0,   1,   0],
       [ 42,   1, 999, ...,   0,   1,   0],
       [ 32,   1, 999, ...,   1,   0,   0]], shape=(12357, 59))

In [23]:
probs_text = predictor.predict(X_test)
probs_text[:50].decode("utf-8").strip().split("\n")

['0.24062314629554749', '0.04597575217485428', '0.10725669']

In [24]:
predictor.delete_endpoint(delete_endpoint_config=True)
print("Endpoint deleted.")

INFO:sagemaker:Deleting endpoint configuration with name: sagemaker-xgboost-2026-10-19-16-57-09-047
INFO:sagemaker:Deleting endpoint with name: sagemaker-xgboost-2026-10-19-16-57-09-047


Endpoint deleted.
